In [ ]:
# Importations de base et Keras/TensorFlow
from google.colab import drive
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.metrics import Recall, AUC # Métriques intégrées utiles
from sklearn.model_selection import train_test_split
from PIL import Image
import os
import numpy as np
import random
import cv2
import matplotlib.pyplot as plt
import time # Pour mesurer le temps

# Importations pour métriques post-entraînement
from scipy.spatial.distance import directed_hausdorff # Pour la distance de Hausdorff

# Montage de Google Drive
drive.mount('/content/drive')

In [ ]:
# Fonctions d'augmentation (rotateImage, bruit, change_gamma, color, random_change)

def rotateImage(image, angle):
    image_center = tuple(np.array(image.shape[1::-1])/2)
    rot_mat = cv2.getRotationMatrix2D(image_center, angle, 1.0)
    return cv2.warpAffine(image, rot_mat, image.shape[1::-1], flags=cv2.INTER_LINEAR)

def bruit(image):
    image_float = image.astype(np.float32)
    noise = np.random.randn(*image.shape) * random.randint(5, 30)
    noisy_image = np.clip(image_float + noise, 0, 255)
    return noisy_image.astype(np.uint8)

def change_gamma(image, alpha=1.0, beta=0.0):
    image_float = image.astype(np.float32)
    adjusted_image = np.clip(alpha * image_float + beta, 0, 255)
    return adjusted_image.astype(np.uint8)

def color(image, alpha=20):
    variation = np.random.randint(-alpha, alpha + 1, size=image.shape, dtype=np.int32)
    colored_image = np.clip(image.astype(np.int32) + variation, 0, 255)
    return colored_image.astype(np.uint8)

def random_change(image):
    img = image.copy()
    if np.random.randint(2):
        img = change_gamma(img, random.uniform(0.8, 1.2), np.random.randint(-50, 50))
    if np.random.randint(2):
        img = bruit(img)
    if np.random.randint(2):
        img = color(img)
    return img

In [ ]:
# --- Fonction Dice Coefficient (Identique à votre code) ---
def dice_coef(y_true, y_pred, smooth=1): # smooth=1 comme dans votre code original
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    # Utilise la formule exacte de votre code original
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

# --- Bloc d'Attention (Votre fonction 'attention_block' originale) ---
def attention_block(x, gating, inter_channels):
    theta_x = layers.Conv2D(inter_channels, (1, 1), strides=(1, 1), padding='same', use_bias=False)(x)
    phi_g = layers.Conv2D(inter_channels, (1, 1), strides=(1, 1), padding='same', use_bias=True)(gating)

    concat = layers.Add()([theta_x, phi_g])
    concat = layers.Activation('relu')(concat)

    psi = layers.Conv2D(1, (1, 1), strides=(1, 1), padding='same', activation='sigmoid', use_bias=True)(concat)
    return layers.Multiply()([x, psi])

# --- Modèle Attention U-Net (Votre fonction 'model_attention' originale) ---
def model_attention(nbr):
    entree = layers.Input(shape=(576, 560, 3), dtype='float32', name='InputImage')

    # Encodeur (Bloc 1)
    result = layers.Conv2D(nbr, 3, activation='relu', padding='same')(entree)
    result = layers.BatchNormalization()(result)
    result = layers.Conv2D(nbr, 3, activation='relu', padding='same')(result)
    result1 = layers.BatchNormalization()(result) # Sortie Bloc 1 pour skip connection
    pool1 = layers.MaxPool2D()(result1)

    # Encodeur (Bloc 2)
    result = layers.Conv2D(2 * nbr, 3, activation='relu', padding='same')(pool1)
    result = layers.BatchNormalization()(result)
    result = layers.Conv2D(2 * nbr, 3, activation='relu', padding='same')(result)
    result2 = layers.BatchNormalization()(result) # Sortie Bloc 2 pour skip connection
    pool2 = layers.MaxPool2D()(result2)

    # Encodeur (Bloc 3)
    result = layers.Conv2D(4 * nbr, 3, activation='relu', padding='same')(pool2)
    result = layers.BatchNormalization()(result)
    result = layers.Conv2D(4 * nbr, 3, activation='relu', padding='same')(result)
    result3 = layers.BatchNormalization()(result) # Sortie Bloc 3 pour skip connection
    pool3 = layers.MaxPool2D()(result3)

    # Encodeur (Bloc 4 - avec 4*nbr filtres comme dans votre original)
    result = layers.Conv2D(4 * nbr, 3, activation='relu', padding='same')(pool3)
    result = layers.BatchNormalization()(result)
    result = layers.Conv2D(4 * nbr, 3, activation='relu', padding='same')(result)
    result4 = layers.BatchNormalization()(result) # Sortie Bloc 4 pour skip connection
    pool4 = layers.MaxPool2D()(result4)

    # Bottleneck (avec 8*nbr puis 4*nbr filtres comme dans votre original)
    result = layers.Conv2D(8 * nbr, 3, activation='relu', padding='same')(pool4)
    result = layers.BatchNormalization()(result)
    result = layers.Conv2D(4 * nbr, 3, activation='relu', padding='same')(result)
    result_bottleneck = layers.BatchNormalization()(result) # Renommé pour clarté

    # --- DEBUT DE VOTRE DECODEUR AVEC ATTENTION ORIGINAL ---
    result = layers.UpSampling2D()(result_bottleneck)
    result = attention_block(result4, result, 4 * nbr) # Attention block comme dans votre code original
    result = layers.Concatenate(axis=3)([result, result4]) # Concaténation comme dans votre code original

    result = layers.UpSampling2D()(result)
    result = attention_block(result3, result, 2 * nbr) # Attention block comme dans votre code original
    result = layers.Concatenate(axis=3)([result, result3]) # Concaténation comme dans votre code original

    result = layers.UpSampling2D()(result)
    result = attention_block(result2, result, nbr) # Attention block comme dans votre code original
    result = layers.Concatenate(axis=3)([result, result2]) # Concaténation comme dans votre code original

    result = layers.UpSampling2D()(result)
    result = attention_block(result1, result, nbr) # Attention block comme dans votre code original
    result = layers.Concatenate(axis=3)([result, result1]) # Concaténation comme dans votre code original
    # --- FIN DE VOTRE DECODEUR AVEC ATTENTION ORIGINAL ---

    # Couche de sortie (Identique à votre code)
    sortie = layers.Conv2D(1, 1, activation='sigmoid', padding='same', name='OutputMask')(result)

    return models.Model(inputs=entree, outputs=sortie)


# --- Nouvelles métriques Keras personnalisées (Identiques au Notebook 1) ---

def iou_coef(y_true, y_pred, smooth=1e-6):
    """Coefficient IoU (Intersection over Union) ou Indice de Jaccard."""
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return iou

# Sensitivity est géré par tf.keras.metrics.Recall(name='sensitivity')

def specificity(y_true, y_pred):
    """Spécificité (True Negative Rate)."""
    neg_y_true = 1 - y_true
    neg_y_pred = 1 - y_pred
    fp = K.sum(K.cast(K.greater(y_pred, 0.5), 'float32') * K.cast(K.equal(y_true, 0), 'float32'))
    tn = K.sum(K.cast(K.less_equal(y_pred, 0.5), 'float32') * K.cast(K.equal(y_true, 0), 'float32'))
    specificity_val = tn / (tn + fp + K.epsilon())
    return specificity_val

In [ ]:
# Chemins vers les données
dir_images = '/content/drive/MyDrive/DRIVE/training/images/'
dir_mask   = '/content/drive/MyDrive/DRIVE/training/1st_manual/'

# Vérification existence dossiers
if not os.path.isdir(dir_images):
    raise ValueError(f"Le dossier d'images n'existe pas : {dir_images}")
if not os.path.isdir(dir_mask):
    raise ValueError(f"Le dossier de masques n'existe pas : {dir_mask}")

tab_images, tab_masks = [], []
target_height, target_width = 576, 560

print("Chargement et augmentation des données (méthode standardisée)...")
# Boucle de chargement et d'augmentation
# (COPIE EXACTE de la boucle du Notebook 1)
# --- DEBUT DE LA BOUCLE COPIEE ---
for fichier in sorted(os.listdir(dir_images)):\
    if fichier.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):\
        try:\
            img_path = os.path.join(dir_images, fichier)\
            # Lire image et recadrer\
            img_orig = cv2.imread(img_path)\
            if img_orig is None: continue \
            img_orig = img_orig[:target_height, :target_width]\
            if img_orig.shape[0] != target_height or img_orig.shape[1] != target_width:\
                img_orig = cv2.resize(img_orig, (target_width, target_height), interpolation=cv2.INTER_AREA)\
\
            # Lire masque et recadrer\
            num = fichier.split('_')[0]\
            file_mask = os.path.join(dir_mask, num + '_manual1.gif')\
            if not os.path.exists(file_mask): continue \
\
            img_mask_orig = np.array(Image.open(file_mask))\
            if len(img_mask_orig.shape) == 3: img_mask_orig = img_mask_orig[:,:,0] \
            img_mask_orig = img_mask_orig[:target_height, :target_width]\
            if img_mask_orig.shape[0] != target_height or img_mask_orig.shape[1] != target_width:\
                 img_mask_orig = cv2.resize(img_mask_orig, (target_width, target_height), interpolation=cv2.INTER_NEAREST)\
\
            # Ajouter l'image et son masque originaux\
            tab_images.append(img_orig)\
            tab_masks.append(img_mask_orig)\
\
            # Boucle d'augmentation (identique à U-Net simple)\
            for angle in [0, 90, 180, 270]: \
                img_r = rotateImage(img_orig, angle)\
                mask_rot_mat = cv2.getRotationMatrix2D(tuple(np.array(img_mask_orig.shape[1::-1])/2), angle, 1.0)\
                img_mask_r = cv2.warpAffine(img_mask_orig, mask_rot_mat, img_mask_orig.shape[1::-1], flags=cv2.INTER_NEAREST)\
\
                for flip_code in [-1, 0, 1]: \
                    for flip in [0, 1]: \
                        img_f = cv2.flip(img_r, flip)\
                        img_mask_f = cv2.flip(img_mask_r, flip)\
\
                        img_augmented = random_change(img_f)\
\
                        img_augmented = img_augmented[:target_height, :target_width]\
                        img_mask_final = img_mask_f[:target_height, :target_width]\
\
                        if img_augmented.shape[0] == target_height and img_augmented.shape[1] == target_width and \
                           img_mask_final.shape[0] == target_height and img_mask_final.shape[1] == target_width:
                              tab_images.append(img_augmented)\
                              tab_masks.append(img_mask_final)\
\
        except Exception as e:\
            print(f"Erreur pendant chargement/augmentation pour {fichier}: {e}")
# --- FIN DE LA BOUCLE COPIEE ---

# Limitation à 500 échantillons si la boucle en génère plus
if len(tab_images) > 500:
    print(f"Attention: Plus de 500 images générées ({len(tab_images)}), limitation à 500.")
    indices = random.sample(range(len(tab_images)), 500)
    tab_images = [tab_images[i] for i in indices]
    tab_masks = [tab_masks[i] for i in indices]

print(f"Chargement et augmentation terminés. Nombre total d'échantillons: {len(tab_images)}")

if not tab_images:
     raise SystemExit("Erreur: Aucune image/masque après chargement/augmentation.")

# Convertir et normaliser
tab_images = np.array(tab_images, dtype=np.float32) / 255.0
tab_masks = np.array(tab_masks, dtype=np.float32) / 255.0

# Assurer que les masques ont une dimension de canal (Batch, H, W, 1) pour Keras
if len(tab_masks.shape) == 3:
    tab_masks = np.expand_dims(tab_masks, axis=-1)
    print(f"Dimension de canal ajoutée aux masques. Nouvelle forme : {tab_masks.shape}")

print(f"Forme finale des images : {tab_images.shape}")
print(f"Forme finale des masques : {tab_masks.shape}")

# Séparer les jeux de données (Même split que pour U-Net simple)
train_images, test_images, train_masks, test_masks = train_test_split(
    tab_images, tab_masks, test_size=0.05, random_state=42
)

print(f"\nDonnées divisées :")
print(f"  Entraînement : {train_images.shape[0]} images, {train_masks.shape[0]} masques")
print(f"  Test (Validation) : {test_images.shape[0]} images, {test_masks.shape[0]} masques")

# Libérer la mémoire
del tab_images, tab_masks

In [ ]:
# Instancier le modèle (Votre fonction model_attention originale)
my_model = model_attention(64)

# Compiler le modèle avec l'optimiseur, la perte ET les mêmes métriques
my_model.compile(optimizer='adam',
                 loss='binary_crossentropy',
                 metrics=['accuracy',
                          dice_coef,
                          iou_coef,
                          Recall(name='sensitivity'),
                          specificity,
                          AUC(name='auc_roc')
                          ])

# Afficher le résumé pour vérifier
print("\nRésumé du modèle Attention U-Net compilé :")
my_model.summary()

In [ ]:
# Cellule 6 (Corrigée) : Entraînement U-Net + Attention avec Callbacks

# --- AJOUT DES CALLBACKS ---
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Définir le chemin pour sauvegarder le meilleur modèle
checkpoint_filepath_attention = '/content/drive/MyDrive/best_unet_attention.h5'

# Callback EarlyStopping: arrête l'entraînement si val_dice_coef ne s'améliore pas pendant 15 époques
# restore_best_weights=True garantit que le modèle aura les meilleurs poids à la fin
early_stopping = EarlyStopping(monitor='val_dice_coef', patience=15, verbose=1, mode='max', restore_best_weights=True)

# Callback ModelCheckpoint: sauvegarde le modèle uniquement lorsque val_dice_coef s'améliore
model_checkpoint = ModelCheckpoint(filepath=checkpoint_filepath_attention,
                                   save_weights_only=False, # Sauvegarde le modèle complet
                                   monitor='val_dice_coef',
                                   mode='max',
                                   save_best_only=True,
                                   verbose=1) # verbose=1 pour voir quand le modèle est sauvegardé

# --- PARAMETRES D'ENTRAINEMENT AJUSTES ---
# Mettre une valeur élevée, EarlyStopping s'occupera d'arrêter au bon moment
EPOCHS = 150
# BATCH_SIZE ajusté pour Colab T4 (compromis mémoire/stabilité) - À TESTER
# DOIT ÊTRE LA MÊME VALEUR que pour le notebook U-Net Simple
BATCH_SIZE = 4

print(f"\n--- Début de l'entraînement ({EPOCHS} époques max, batch size {BATCH_SIZE}) ---")
print(f"Early Stopping surveille '{early_stopping.monitor}' ({early_stopping.mode}) avec patience {early_stopping.patience}.")
# --- LIGNE PRINT CORRIGÉE CI-DESSOUS ---
print(f"Model Checkpoint sauvegardera le meilleur modèle selon '{model_checkpoint.monitor}' dans '{checkpoint_filepath_attention}'.")

start_training_time = time.time()

# Entraîner le modèle avec les callbacks
# Utilise les données augmentées et splitées (train_images, test_images...)
history = my_model.fit(train_images, train_masks,
                       epochs=EPOCHS,
                       batch_size=BATCH_SIZE,
                       validation_data=(test_images, test_masks), # Utilise le même set de validation
                       callbacks=[early_stopping, model_checkpoint], # AJOUT DES CALLBACKS ICI
                       verbose=1) # verbose=1 pour voir la progression

end_training_time = time.time()
training_runtime = end_training_time - start_training_time
print(f"\n--- Entraînement terminé (potentiellement avant {EPOCHS} époques grâce à EarlyStopping) ---")
print(f"Temps d'exécution total de l'entraînement (Runtime) : {training_runtime:.2f} secondes ({training_runtime/60:.2f} minutes)")
print(f"Le meilleur modèle a été sauvegardé à : {checkpoint_filepath_attention}")
# Note: Grâce à restore_best_weights=True dans EarlyStopping, 'my_model' contient maintenant les meilleurs poids.

In [ ]:
print("\n--- Génération des graphiques de l'historique d'entraînement ---")

history_dict = history.history

metrics_to_plot = {
    'loss': 'Loss',
    'accuracy': 'Accuracy',
    'dice_coef': 'Dice Coefficient',
    'iou_coef': 'IoU (Jaccard)',
    'sensitivity': 'Sensitivity (Recall)',
    'specificity': 'Specificity',
    'auc_roc': 'AUC-ROC'
}

num_metrics = len(metrics_to_plot)
num_cols = 3
num_rows = (num_metrics + num_cols - 1) // num_cols

plt.style.use('ggplot')
fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(num_cols * 6, num_rows * 4.5))
axes = axes.flatten()

plot_index = 0
for metric_key, metric_name in metrics_to_plot.items():
    if plot_index >= len(axes): break # Sécurité
    ax = axes[plot_index]
    if metric_key in history_dict:
        epochs_range = range(1, len(history_dict[metric_key]) + 1) # Pour l'axe X
        ax.plot(epochs_range, history_dict[metric_key], label=f'Train {metric_name}', marker='.')
        val_metric_key = f'val_{metric_key}'
        if val_metric_key in history_dict:
            ax.plot(epochs_range, history_dict[val_metric_key], label=f'Validation {metric_name}', marker='.')
        else:
            print(f"Note : Pas de données de validation trouvées pour '{val_metric_key}'")
        ax.set_title(f'{metric_name} vs. Epochs')
        ax.set_xlabel('Epochs')
        ax.set_ylabel(metric_name)
        ax.legend()
        plot_index += 1
    else:
        print(f"Attention : Métrique '{metric_key}' non trouvée dans l'historique.")

# Cacher axes non utilisés
for i in range(plot_index, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

# Affichage valeurs finales (ATTENTION: celles de la dernière époque, pas forcément les meilleures si EarlyStopping sans restore_best_weights)
# Si restore_best_weights=True, ces valeurs finales correspondent à la meilleure époque arrêtée
print("\n--- Valeurs finales des métriques (dernière époque effectuée / meilleure époque si restore_best_weights=True) ---")
final_epoch = len(history_dict.get('loss', []))
if final_epoch > 0:
  for metric_key, metric_name in metrics_to_plot.items():
      print(f"--- {metric_name} ---")
      train_val = history_dict.get(metric_key, [np.nan])[-1] # Prend la dernière valeur ou NaN
      val_val = history_dict.get(f'val_{metric_key}', [np.nan])[-1] # Prend la dernière valeur ou NaN
      print(f"  Train      : {train_val:.4f}")
      print(f"  Validation : {val_val:.4f}")
else:
  print("L'entraînement n'a pas produit d'historique.")

In [ ]:
print("\n--- Évaluation post-entraînement sur l'ensemble de validation ---")
# Note: Utilise test_images/test_masks qui ont servi de validation_data durant fit()
# 'my_model' a les meilleurs poids si restore_best_weights=True dans EarlyStopping

# 1. Mesurer le Temps d'Inférence (Inference Time) sur l'ensemble de validation
num_val_samples = len(test_images) # Sera 25 avec les données standardisées
if num_val_samples > 0:
    print(f"Calcul du temps d'inférence sur {num_val_samples} images de validation...")
    start_inference_time = time.time()
    inference_batch_size = min(BATCH_SIZE, num_val_samples)
    predictions_prob_val = my_model.predict(test_images, batch_size=inference_batch_size)
    end_inference_time = time.time()

    total_inference_time = end_inference_time - start_inference_time
    avg_inference_time_per_image = total_inference_time / num_val_samples
    print(f"  Temps d'inférence total pour {num_val_samples} images: {total_inference_time:.4f} secondes")
    print(f"  Temps d'inférence moyen par image: {avg_inference_time_per_image:.6f} secondes")

    # 2. Calculer la Distance de Hausdorff sur l'ensemble de validation
    print(f"\nCalcul de la distance de Hausdorff pour {num_val_samples} paires de masques...")
    predictions_binary_val = (predictions_prob_val > 0.5).astype(np.uint8)
    test_masks_binary_val = (test_masks > 0.5).astype(np.uint8)

    hausdorff_distances = []
    for i in range(num_val_samples):
        gt_mask = test_masks_binary_val[i].squeeze()
        pred_mask = predictions_binary_val[i].squeeze()
        coords_gt = np.argwhere(gt_mask > 0)
        coords_pred = np.argwhere(pred_mask > 0)

        if coords_gt.shape[0] > 0 and coords_pred.shape[0] > 0:
            try:
                hd1 = directed_hausdorff(coords_gt, coords_pred)[0]
                hd2 = directed_hausdorff(coords_pred, coords_gt)[0]
                hausdorff_distances.append(max(hd1, hd2))
            except Exception as e:
                print(f" Avertissement: Erreur calcul Hausdorff pour l'image {i}: {e}")
                hausdorff_distances.append(np.nan)
        elif coords_gt.shape[0] == 0 and coords_pred.shape[0] == 0:
            hausdorff_distances.append(0.0)
        else:
            hausdorff_distances.append(np.nan)

    valid_distances = [d for d in hausdorff_distances if not np.isnan(d)]
    if valid_distances:
        avg_hausdorff = np.mean(valid_distances)
        std_hausdorff = np.std(valid_distances)
        print(f"\n  Distance de Hausdorff Moyenne (Validation Set, {len(valid_distances)} paires valides): {avg_hausdorff:.4f}")
        print(f"  Distance de Hausdorff Écart-type (Validation Set, {len(valid_distances)} paires valides): {std_hausdorff:.4f}")
        print("  (Note: La métrique 's2s distance' est souvent liée à Hausdorff ou ASD.)")
    else:
        print("\n  Aucune distance de Hausdorff valide n'a pu être calculée sur le set de validation.")

else:
    print("Ensemble de validation vide. Métriques post-entraînement non calculées.")

In [ ]:
# Enregistrer l'état final du modèle
model_save_path_final = '/content/drive/MyDrive/final_unet_attention.h5'
my_model.save(model_save_path_final)
print(f"\nModèle final (U-Net Attention) enregistré sous : {model_save_path_final}")
print(f"Le MEILLEUR modèle basé sur '{model_checkpoint.monitor}' est dans : {checkpoint_filepath_attention}")

print("\n--- Fin du script U-Net Attention ---")